# Speech To Text with LLM

## Feature Extraction - Audio To Text

### Environment Preparation

In [ ]:
# Import necessary libraries and modules

# Operating System module - provides functions for interacting with the operating system
import os

# IPython display module to play audio in Jupyter notebooks
from IPython.display import Audio

# Hugging Face library for datasets - helps load and process datasets
from datasets import load_dataset

# Transformer library to load and use the Whisper model for audio-to-text tasks
from transformers import WhisperProcessor, WhisperForConditionalGeneration

# Librosa - a library for analyzing audio and music; provides audio processing tools
import librosa

# PyTorch - a deep learning framework used for managing model computations and GPU compatibility
import torch

# System-specific parameters and functions, e.g., to modify the Python path
import sys


# Set cache locations for Hugging Face and datasets

# Retrieve the Hugging Face cache directory from the environment variable and convert it to an absolute path.
# This path is used to store pre-trained models to avoid downloading them multiple times.
huggingface_cache_location = r"../../resources/huggingface_cache"

# # Retrieve the datasets cache directory from the environment variable and convert it to an absolute path.
# # This is the location where datasets are cached after initial download, helping save bandwidth and time.
# datasets_cache_location = os.path.abspath("../../../" + os.getenv("datasets_cache_location"))

# Set up the parent directory for custom modules

# Define the current directory path. This is useful if you have a custom module or scripts located in a current
# directory that you want to import into this script.
current_dir = os.path.abspath('./')

# Add the parent directory path to the Python path (sys.path), which allows importing modules from it.
sys.path.append(current_dir)

# Import any custom utilities located in the parent directory
# This step is optional and depends on whether you have additional scripts, such as helper functions, in the parent directory.
import local_utilities

data_helper = local_utilities.DataHelper()

# Set device to GPU if available, otherwise default to CPU

# Check if a CUDA-compatible GPU is available, and if so, set the device to GPU.
# This enables faster model processing by utilizing the GPU for heavy computations.
# If no GPU is available, the device will be set to CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


### Function Definitions

In [ ]:
import numpy as np
import pandas as pd
import psychai.feature.feature_extraction.feature_processor
import cv2
import matplotlib.pyplot as plt
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas


class SpeechToTextExtractor:

    def __init__(self, processor, model):
        self.processor = processor
        self.model = model
        pass

    # def summarized_process_folder(self,input_file_fullname):

    #     # Extract features using multiple strategies and plot them
    #     strategies = ['mean', 'variance', 'max', 'end_to_begin']
        
    #     try:
    #         features, feature_names = self.process_audio_with_file_path(input_file_fullname)
    #     except Exception as e:
    #         print(f"Error: {e}")
    #         return None, None   

    #     # Summarize features using the specified strategies
    #     processor = psychai.feature.feature_extraction.feature_processor.FeatureProcessor()
    #     #processor = FeatureProcessor()
    #     summarized_features, summarized_feature_names = processor.summarize_features(features, strategies, feature_names)

    #     return summarized_features, summarized_feature_names  

    def process_audio_with_file_path(self, file_path):

        # Load WAV file
        # Load audio file using librosa.load function. Set sr=None to preserve the original sampling rate of the audio
        signal, sr = librosa.load(file_path, sr=None)  # sr=None preserves original sample rate

        # Resample the audio signal
        # Resample the audio signal from the original sampling rate to 16000Hz, as the Whisper model requires a fixed sampling rate
        resampled_signal = librosa.resample(signal, orig_sr=sr, target_sr=16000)

        return self.process_audio_as_array(resampled_signal)

    def process_audio_as_array(self, sample):

        # Process audio samples with the processor, passing in the audio array and sample rate, then convert the result to a PyTorch tensor
        input_features = self.processor(sample, sampling_rate=16000, return_tensors="pt").input_features 

        # Use the model to generate predicted token ids
        predicted_ids = self.model.generate(input_features)
        # Decode token ids to text
        transcription = self.processor.batch_decode(predicted_ids, skip_special_tokens=False)
        # Print transcribed text
        print(transcription)

        return np.array([transcription]), ['transcription']

    def process_audio_from_datasets(self, sample):

        # Process audio samples with the processor, passing in the audio array and sample rate, then convert the result to a PyTorch tensor
        input_features = self.processor(sample["array"], sampling_rate=sample["sampling_rate"], return_tensors="pt").input_features 

        # Use the model to generate predicted token ids
        predicted_ids = self.model.generate(input_features)
        # Decode token ids to text
        transcription = self.processor.batch_decode(predicted_ids, skip_special_tokens=False)
        # Print transcribed text
        print(transcription)

        return transcription

        # # Process resampled audio signal into model input features
        # # Process the resampled signal with the Whisper processor and set the sampling rate to 16000Hz. return_tensors="pt" indicates that the returned value is a PyTorch tensor
        # input_features = self.processor(signal, sampling_rate=16000, return_tensors="pt").input_features 

        # # Generate token ID
        # # Use the model's generate method to produce predicted token IDs from input features
        # predicted_ids = self.model.generate(input_features)

        # # Decode token ID to text
        # # Use the processor's batch_decode method to decode predicted token IDs into text, skip_special_tokens=False indicates not skipping special tokens
        # transcription = self.processor.batch_decode(predicted_ids, skip_special_tokens=False)

        # # Print transcription results
        # return transcription

# # Define a function to clean up the text in each row
# def clean_text(text):
#     # Check if both markers exist in the text
#     start_marker = "<|notimestamps|>"
#     end_marker = "<|endoftext|>"
    
#     if start_marker in text and end_marker in text:
#         # Find the start and end positions of the markers
#         start_idx = text.find(start_marker) + len(start_marker)
#         end_idx = text.find(end_marker)
        
#         # Extract the text between the markers
#         return text[start_idx:end_idx].strip()  # .strip() to remove any leading/trailing spaces
#     else:
#         # If markers are not found, return the original text
#         return text
    
# # Group by user_id
# def calculate_mean_by_user_id(df):
#     # Step 1: Check consistency in non-numerical columns
#     df_consistent = df.groupby('user_id').apply(check_non_numerical_consistency).reset_index(drop=True)

#     # Step 2: Calculate mean for numerical columns
#     df_mean = df.groupby('user_id').mean(numeric_only=True).reset_index()

#     # Step 3: Add back consistent non-numerical columns
#     # Ensure 'user_id' is present in non_numerical_columns subset
#     non_numerical_columns = df_consistent[['user_id'] + df_consistent.select_dtypes(include=['object']).columns.tolist()].drop_duplicates(subset=['user_id'])
    
#     df_mean = pd.merge(df_mean, non_numerical_columns, on='user_id', how='left')

#     return df_mean


# # Group by user_id
# def calculate_mean_by_user_id(df):
#     # Step 1: Check consistency in non-numerical columns
#     df_consistent = df.groupby('user_id').apply(check_non_numerical_consistency).reset_index(drop=True)

#     # Step 2: Calculate mean for numerical columns
#     df_mean = df.groupby('user_id').mean(numeric_only=True).reset_index()

#     # Step 3: Add back consistent non-numerical columns
#     # Ensure 'user_id' is present in non_numerical_columns subset
#     non_numerical_columns = df_consistent[['user_id'] + df_consistent.select_dtypes(include=['object']).columns.tolist()].drop_duplicates(subset=['user_id'])
    
#     df_mean = pd.merge(df_mean, non_numerical_columns, on='user_id', how='left')

#     return df_mean

# # Function to check consistency in non-numerical columns
# def check_non_numerical_consistency(group):
#     for col in group.select_dtypes(include=['object']).columns:
#         if group[col].nunique() > 1:
#             raise ValueError(f"Inconsistent values found in column '{col}' for user_id '{group.name}'")
#     return group.iloc[0]  # Return the first row for consistent non-numerical columns


### Initialization

In [ ]:
# Load Whisper processor and model from pretrained model
processor = WhisperProcessor.from_pretrained("openai/whisper-large-v2", cache_dir=huggingface_cache_location)
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-large-v2", cache_dir=huggingface_cache_location)
model.config.forced_decoder_ids = None  # Ensure the model does not force the use of decoder ID

# Load a sample dataset
ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation", cache_dir=datasets_cache_location)



In [ ]:
speech_to_text_extractor = SpeechToTextExtractor(processor, model)

### Audio To Text

#### English Example

This code first processes the first audio sample in the dataset, converting it to a format acceptable by the model, then uses the Whisper model to make predictions and decodes the prediction results (token ids) into readable text form. Finally, it prints out the transcribed text. Note that the skip_special_tokens=False parameter controls whether to skip special tokens; you can adjust it as needed during decoding.

In [ ]:
# Get audio information for the first sample
sample = ds[0]["audio"]
# Process audio samples with the processor, passing in the audio array and sample rate, then convert the result to a PyTorch tensor
input_features = processor(sample["array"], sampling_rate=sample["sampling_rate"], return_tensors="pt").input_features 

# Use the model to generate predicted token ids
predicted_ids = model.generate(input_features)
# Decode token ids to text
transcription = processor.batch_decode(predicted_ids, skip_special_tokens=False)
# Print transcribed text
print(transcription)

In [ ]:
# Get audio information for the first sample
sample = ds[0]["audio"]
transcription = speech_to_text_extractor.process_audio_from_datasets(sample)

#### Chinese Example

In [ ]:
# Correct path to your audio file
audio_file_path = r"../resources/raw_data/005/Segment_Audio/SUBJ_001_example.wav"
IPython.display.Audio(audio_file_path)

In [ ]:
# Load WAV file
# Load audio file using librosa.load function. Set sr=None to preserve the original sampling rate of the audio
signal, sr = librosa.load(audio_file_path, sr=None)  # sr=None preserves original sample rate

# Resample the audio signal
# Resample the audio signal from the original sampling rate to 16000Hz, as the Whisper model requires a fixed sampling rate
resampled_signal = librosa.resample(signal, orig_sr=sr, target_sr=16000)

# Process resampled audio signal into model input features
# Process the resampled signal with the Whisper processor and set the sampling rate to 16000Hz. return_tensors="pt" indicates that PyTorch tensors are returned
input_features = processor(resampled_signal, sampling_rate=16000, return_tensors="pt").input_features 

# Generate token ID
# Use the model's generate method to produce predicted token IDs from input features
predicted_ids = model.generate(input_features)

# Decode token id to text
# Use the processor's batch_decode method to decode predicted token IDs into text, skip_special_tokens=False indicates not skipping special tokens
transcription = processor.batch_decode(predicted_ids, skip_special_tokens=False)

# Print transcription results
print(transcription)

In [ ]:
transcription = speech_to_text_extractor.process_audio_with_file_path(audio_file_path)

#### File Folder and Class Example

In [ ]:
#base_directory = "../resources/raw_data/example/thermal_segment.csv"
base_directory = "../resources/raw_data"

# Create a FileFilter instance with multiple attribute filters
filters = [(2, '25'),(2, '26'),(2, '27'),(2, '28')]
file_filter = utilities.FileFilter(base_directory, attribute_filters=filters, modality_value='Segment_Audio')

# Get matching files for user IDs from 1 to 13
df = file_filter.get_matching_files(range(1, 120))

# Load the rct.csv file
rct_df = pd.read_csv('../resources/data/rct/rct.csv')

# Merge the two dataframes based on the 'user_id' column
df_updated = pd.merge(df, rct_df[['user_id', 'Group']], on='user_id', how='left')

# Find the row with the largest file_size for each user_id
# df_largest = df_updated.loc[df_updated.groupby('user_id')['file_size'].idxmax()]
df_largest = df_updated.loc[df_updated.groupby(['user_id', 'attribute_value'])['file_size'].idxmax()]

# Reset the index if needed
df_largest.reset_index(drop=True, inplace=True)

# Display the result
print(df_largest)

In [ ]:
import os
import pandas as pd
import pickle
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import time

# Path to cache folder
cache_folder = "../resources/raw_data/example/thermal_segment.csv"
os.makedirs(cache_folder, exist_ok=True)  # Ensure cache folder exists

# Function to process each file with caching
def process_file(row):
    audio_file_path = row['files']
    cache_file_path = os.path.join(cache_folder, f"{os.path.basename(audio_file_path)}.pkl")
    
    # Check if the cache file exists
    if os.path.exists(cache_file_path):
        # Load the cached result
        with open(cache_file_path, 'rb') as cache_file:
            text = pickle.load(cache_file)
    else:
        # Run the speech-to-text processing and save to cache
        text = speech_to_text_extractor.process_audio_with_file_path(audio_file_path)
        text = clean_text(text)
        
        # Save the result to a cache file
        with open(cache_file_path, 'wb') as cache_file:
            pickle.dump(text, cache_file)

    # Return the text result for adding to DataFrame
    return text

# Apply parallel processing with progress tracking
def process_files_in_parallel(df):
    # Add a "text" column to the DataFrame to store results
    df['text'] = None
    
    # Use ThreadPoolExecutor for parallel processing
    with ThreadPoolExecutor() as executor:
        futures = {executor.submit(process_file, row): idx for idx, row in df.iterrows()}
        
        # Use tqdm for a progress bar
        for future in tqdm(as_completed(futures), total=len(futures), desc="Processing files"):
            idx = futures[future]
            try:
                print(f"[file {idx}] starts.")
                # Update the 'text' column with the result
                df.at[idx, 'text'] = future.result()
                print(f"[file {idx}] completed.")
            except Exception as e:
                print(f"Error processing file at index {idx}: {e}")

    return df

# Run the processing function
df_largest = process_files_in_parallel(df_largest)
df_largest.to_csv("../resources/raw_data/example/thermal_segment.csv")

## Footnote
- Copyright (c) 2026 Ivan Liu. Licensed under CC BY 4.0.
- Last Update: 2024
- Env: psychai241104
- References: 
    - None